# 03 · Name frequency (exploration)

> **Data:** exploration on the **uncleaned** in-scope pairs (`pairs_scoped.parquet`), fixed on purpose (`C.PAIRS_SCOPED`, not `C.PAIRS_INPUT`). Kept as documentation of how the features were built and chosen; not re-run on the cleaned data, so the numbers in the conclusions stay valid. Final numbers come from notebook 07.

**Goal:** build frequency tables for the name-frequency features and check whether they explain false alarms.

**Approach:** in the OpenSanctions targets export (`targets.simple`), first collapse duplicate entities (section 1b), then count once per entity, separately for persons and for companies/organisations:
- **words** ("thomas", "muller") — section 2
- **full names** ("muller thomas", word order ignored) — section 2b

**Status:** exploration only. Nothing is written to `data/`; the tables live as variables in this notebook. Once we are happy, the code moves to `src/` and `scripts/`.

Runtime: about 4–6 minutes for the whole notebook. Kernel: *Python (ds_sanctions)*.

In [ ]:
import sys, re, time
sys.path.insert(0, "../src")

from collections import Counter
import numpy as np
import pandas as pd
from unidecode import unidecode
from screening import config as C

TARGETS_CSV = C.DATA_RAW / "targets.simple-20261111.csv"   # downloaded 2026-11-11
GROUPS = {
    "person": ("Person",),
    "org": ("Company", "Organization", "LegalEntity"),
}
MIN_LEN = 2   # drop one-letter words (initials)

targets = pd.read_csv(TARGETS_CSV, usecols=["id", "schema", "name", "aliases", "birth_date", "dataset"],
                      dtype=str, keep_default_na=False)
print(f"{len(targets):,} target entities")
targets["schema"].value_counts()

**Conclusion.**
- **Persons dominate:** 1,036,156 of the 1,248,755 entities. Companies (100,010), legal entities (58,888) and organisations (9,755) follow.
- **Securities, crypto wallets and vessels** have names too, but are not part of our pairs.

**Next step:** count persons and organisations (company, organisation, legal entity) in separate tables, since common words differ completely between them.

## 1 · Normalisation

Every name is transliterated to ASCII (`unidecode`), lowercased, and everything that is not a letter or digit becomes a space. The **same function** must later be used for the pairs, otherwise "müller" in a pair would not find "muller" in the table.

In [ ]:
_NON_ALNUM = re.compile(r"[^a-z0-9]+")

def normalize(name):
    if not name:
        return ""
    return " ".join(_NON_ALNUM.sub(" ", unidecode(name).lower()).split())

def tokens(name, min_len=MIN_LEN):
    return [t for t in normalize(name).split() if len(t) >= min_len]

def entity_tokens(name, aliases, min_len=MIN_LEN):
    """Set of words over the name and all ';'-separated aliases of one entity."""
    names = [name] + ([a for a in aliases.split(";") if a.strip()] if aliases else [])
    return {t for n in names for t in tokens(n, min_len)}

examples = ["Thomas Müller", "THOMAS MULLER", "Олександр Петренко", "عبد الكريم",
            "王伟", "J. Smith", "Myanmar Yatai International Holding Group Co., Ltd.",
            'ООО "Газпром добыча Ноябрьск"']
pd.DataFrame({"original": examples, "normalized": [normalize(e) for e in examples]})

**Conclusion.**
- **The normalisation does what it should:** accents, upper/lower case and Cyrillic or Chinese script all map to the same ASCII form ("Thomas Müller" = "THOMAS MULLER"; "王伟" = "wang wei").
- **Arabic loses its vowels and the letter ʿayn** ("عبد الكريم" → "bd lkrym"), so Arabic names written in Latin and in Arabic script do not match. That is a known limit of `unidecode`.

**Next step:** none for frequency; cross-script pairs are handled by `n_script` in notebook 04.

## 1b · Collapse duplicate entities

Some people appear as many separate entities that OpenSanctions has not merged (e.g. "Morten Austestad": 109 entities with the same birth date, from *Norway State-Owned Enterprises Leadership*). Counting them separately makes a unique name look very common.

Rule: two entities of the same schema are treated as **one person** if their primary names have the same key (normalised, word order ignored) **and** both have the same birth date.

Entities without a birth date are never collapsed. A rule "same name + same source, no birth date" was tested and rejected: it merged distinct people with common names (e.g. 21 × "Muhammad Aslam" on the Pakistan NACTA list, 18 × "José Carlos da Silva" among Brazilian PEPs), exactly the names whose frequency this feature should capture.

All counts below use the collapsed table `entities`.

In [ ]:
primary_key = targets["name"].map(lambda n: " ".join(sorted(tokens(n, min_len=1))))
has_dob = targets["birth_date"].str.strip() != ""
# with birth date: name key + birth date; without: the entity's own id (never collapsed)
targets["identity"] = np.where(has_dob,
                               primary_key + " | dob " + targets["birth_date"],
                               "id " + targets["id"])

entities = targets.drop_duplicates(subset=["schema", "identity"]).copy()
removed = (targets["schema"].value_counts() - entities["schema"].value_counts()).fillna(0).astype(int)
print(f"{len(targets):,} entities -> {len(entities):,} after collapsing ({len(targets) - len(entities):,} removed)")
display(pd.DataFrame({"before": targets["schema"].value_counts(), "after": entities["schema"].value_counts(),
                      "removed": removed}).sort_values("before", ascending=False).head(8))

# identities that were collapsed most often
dup = targets[has_dob].groupby(["schema", "identity"]).size().sort_values(ascending=False)
display(dup[dup > 1].head(20).rename("entities collapsed into one").to_frame())

# the two names that triggered this check
check = targets[primary_key.isin(["austestad morten", "jorn seljelid steinar"])]
print(f"{len(check)} rows for the two names; distinct birth dates: {check['birth_date'].nunique()}, distinct sources: {check['dataset'].nunique()}")
check[["schema", "name", "birth_date", "dataset"]].drop_duplicates().head(20)

**Conclusion.**
- **Collapsing removes 34,422 person entities** (3.3%) and no companies.
- **Almost all of it comes from two Norwegian board members** listed 109 and 77 times with the same birth date and source. Without collapsing, their names would look very common.
- **The rest are groups of 2–3 entities** with the same name and birth date: plausible duplicates.

**Next step:** use the collapsed table for all counts.

## 2 · Count words once per entity

For each entity: the **set** of its words across name and aliases. Then count in how many entities each word appears.

- 5 different people called Thomas Müller → "thomas" +5
- 1 person with 5 spellings → each word +1 only

Result per group: a table with `token`, `n_entities` (number of entities whose names contain the word) and `share` (n_entities / entities in the group).

In [ ]:
def count_tokens(df):
    counts = Counter()
    for name, aliases in zip(df["name"], df["aliases"]):
        counts.update(entity_tokens(name, aliases))
    return counts

freq, n_entities = {}, {}
for group, schemas in GROUPS.items():
    sub = entities[entities["schema"].isin(schemas)]
    t0 = time.time()
    counts = count_tokens(sub)
    table = (pd.DataFrame(list(counts.items()), columns=["token", "n_entities"])
               .sort_values("n_entities", ascending=False, ignore_index=True))
    table["share"] = table["n_entities"] / len(sub)
    freq[group], n_entities[group] = table, len(sub)
    print(f"{group:6s} {len(sub):>10,} entities  {len(table):>10,} distinct words  ({time.time() - t0:.0f}s)")

lookup = {g: dict(zip(t["token"], t["n_entities"])) for g, t in freq.items()}
freq["person"].head(10)

**Conclusion.** The commonest person words are particles and very common first and last names (de, jose, maria, da, silva, li). The person table has 713,601 distinct words over 1,001,734 entities, so most words are rare. Counting takes under a minute.

**Next step:** look at the distribution (section 3).

## 2b · Count full names once per entity

Same idea for the whole name. The key is the **sorted** normalised words, so "Thomas Müller", "MÜLLER, Thomas" and "Muller Thomas" are one name: `muller thomas`. Here one-letter words are kept (min length 1), so "J. Smith" and "John Smith" stay different names.

Result per group: `full_name`, `n_entities` (number of entities carrying this name or alias).

In [ ]:
def name_key(name):
    """Order-insensitive key of one full name; '' if nothing is left."""
    return " ".join(sorted(tokens(name, min_len=1)))

def entity_name_keys(name, aliases):
    names = [name] + ([a for a in aliases.split(";") if a.strip()] if aliases else [])
    return {k for k in (name_key(n) for n in names) if k}

full_freq = {}
for group, schemas in GROUPS.items():
    sub = entities[entities["schema"].isin(schemas)]
    t0 = time.time()
    counts = Counter()
    for name, aliases in zip(sub["name"], sub["aliases"]):
        counts.update(entity_name_keys(name, aliases))
    table = (pd.DataFrame(list(counts.items()), columns=["full_name", "n_entities"])
               .sort_values("n_entities", ascending=False, ignore_index=True))
    full_freq[group] = table
    shared = (table["n_entities"] >= 2).mean()
    print(f"{group:6s} {len(table):>10,} distinct full names, {shared:.1%} carried by 2+ entities  ({time.time() - t0:.0f}s)")

full_lookup = {g: dict(zip(t["full_name"], t["n_entities"])) for g, t in full_freq.items()}

display(pd.concat({g: t.head(25).reset_index(drop=True) for g, t in full_freq.items()}, axis=1))

name_bands = [0, 2, 5, 20, 10**7]
name_labels = ["1-2", "3-5", "6-20", ">20"]
display(pd.concat({g: pd.cut(t["n_entities"], name_bands, labels=name_labels).value_counts().sort_index()
                   for g, t in full_freq.items()}, axis=1))

probe_names = ["Thomas Müller", "Ivan Ivanov", "David Smith", "John Smith", "Abdul Karim",
               "Abdul Kareem", "Muhammad Ali", "Li Wei", "Ilya Eduardovich Torosov"]
pd.DataFrame({"name": probe_names, "key": [name_key(n) for n in probe_names],
              "n_entities": [full_lookup["person"].get(name_key(n), 0) for n in probe_names]})

**Conclusion.**
- **Full names are much more specific than words:** only 3.3% of person full names and 1.3% of organisation full names are carried by two or more entities.
- **The most common person full names are Arabic noms de guerre** (abu omar, abu ali …) and transliteration artefacts ("bw mr" from Arabic script). These are exactly the names a screening system should treat as weak evidence.
- **Very common full names (more than 20 entities) are rare:** 43 person and 3 organisation names.

**Next step:** use both the rarest shared word and the rarest shared full name as features (section 5 tests whether both add something).

In [ ]:
# effect of collapsing on the most shared full names: rows in the raw targets file vs after collapsing
top = full_freq["person"].head(15)["full_name"].tolist() + ["austestad morten", "jorn seljelid steinar"]
pd.DataFrame({"full_name": top,
              "raw_rows_primary_name": [(primary_key[targets["schema"] == "Person"] == k).sum() for k in top],
              "entities_after_collapse": [full_lookup["person"].get(k, 0) for k in top]}).drop_duplicates("full_name")

**Conclusion.** Counting entities instead of raw rows changes the ranking where it should: "austestad morten" drops from 109 to 1 (duplicates collapsed), while alias-only names such as "bw mr" (0 as a primary name, 65 entities via aliases) appear, because aliases are counted too.

**Next step:** none.

## 3 · What the tables look like

In [ ]:
# most common words per group
pd.concat({g: t.head(30).reset_index(drop=True) for g, t in freq.items()}, axis=1)

**Conclusion.**
- **Organisation words are dominated by legal forms:** the Russian "obshchestvo s ogranichennoi otvetstvennost iu" (LLC) appears in 21% of all organisations; ltd, company, co and limited follow.
- **Chinese syllables are also common** (xian, gong, shi, xin).
- **Such words carry almost no evidence,** which the "rarest shared word" automatically skips as long as a rarer word is shared.

**Next step:** no stop list needed for the frequency feature; legal forms get their own feature (`n_legal_form`).

In [ ]:
# how many words fall into each frequency band
bands = [0, 1, 2, 10, 100, 1000, 10**7]
labels = ["1", "2", "3-10", "11-100", "101-1000", ">1000"]
pd.concat({g: pd.cut(t["n_entities"], bands, labels=labels).value_counts().sort_index()
           for g, t in freq.items()}, axis=1)

**Conclusion.**
- **Most words are rare:** 486,145 person words (68%) occur in one entity only, and only 456 in more than 1,000.
- **The heavy tail is why bands work better than raw counts:** a few very common words would dominate any linear scale.

**Next step:** bands 0–2 / 3–10 / 11–100 / 101–1000 / > 1000, with 0 merged into 1–2 (merge effect, notebook 06).

In [ ]:
# probe words
probe_person = ["thomas", "muller", "abdul", "kareem", "karim", "ali", "muhammad",
                "ivanov", "petrenko", "torosov", "smith", "wang", "li", "kim"]
probe_org = ["ltd", "llc", "ooo", "obshchestvo", "gmbh", "company", "bank", "gazprom"]
display(pd.DataFrame({"token": probe_person, "n_entities": [lookup["person"].get(t, 0) for t in probe_person]}))
display(pd.DataFrame({"token": probe_org, "n_entities": [lookup["org"].get(t, 0) for t in probe_org]}))

**Conclusion.**
- **The counts match intuition:** muhammad 5,211, ali 4,921, thomas 4,451 entities, against petrenko 31 and torosov 1.
- **Legal forms are common in the organisation table:** obshchestvo 34,586, ltd 15,472.

**Next step:** none; the table is plausible.

## 4 · Cross-check with names.txt

`names.txt` counts *spellings* (each distinct name string once), not entities, and mixes all schemas. If both sources rank words similarly, the feature is robust to the choice of source. Here each word is counted once per line of names.txt.

In [ ]:
with open(C.NAMES_TXT, encoding="utf-8") as f:
    names_txt = Counter(t for line in f for t in set(tokens(line.rstrip("\n"))))

all_entities = pd.concat(freq.values()).groupby("token", as_index=False)["n_entities"].sum()
cmp = all_entities.merge(pd.DataFrame(list(names_txt.items()), columns=["token", "n_names_txt"]),
                         on="token", how="inner")
print(f"words in both sources: {len(cmp):,}")
print("Spearman rank correlation:", round(cmp["n_entities"].corr(cmp["n_names_txt"], method="spearman"), 3))
cmp.assign(spellings_per_entity=cmp["n_names_txt"] / cmp["n_entities"]).sort_values("n_entities", ascending=False).head(20)

**Conclusion.**
- **The two sources rank words fairly similarly** (Spearman 0.783 over 820,900 shared words).
- **But `names.txt` inflates some words:** it counts spellings, not entities. "ltd" has 6 spellings per entity, "of" even 16 (also from positions and other schemas).

**Next step:** check whether the disagreement is only noise among rare words (4b).

### 4b · Is the disagreement only noise among rare words?

The overall Spearman value is pulled down by ties: most words occur only once or twice, where the ranking is random. For the feature, what matters is whether both sources put a word in the **same band**.

In [ ]:
# 1) correlation restricted to words that are not rare
for k in [2, 5, 10, 50]:
    sub = cmp[cmp["n_entities"] >= k]
    print(f"words with >= {k:>2} entities: {len(sub):>9,}   Spearman {sub['n_entities'].corr(sub['n_names_txt'], method='spearman'):.3f}")

# 2) same band in both sources?
band_edges = [0, 2, 10, 100, 1000, 10**8]
band_names = ["1-2", "3-10", "11-100", "101-1000", ">1000"]
b_t = pd.cut(cmp["n_entities"], band_edges, labels=band_names)
b_n = pd.cut(cmp["n_names_txt"], band_edges, labels=band_names)
same = (b_t == b_n).mean()
adjacent = (abs(b_t.cat.codes - b_n.cat.codes) <= 1).mean()
print(f"\nsame band: {same:.1%}   same or neighbouring band: {adjacent:.1%}")
display(pd.crosstab(b_t.rename("band in targets.simple"), b_n.rename("band in names.txt")))

**Conclusion.**
- **The agreement rises for words that occur more often:** Spearman 0.823 at 2+ entities, 0.866 at 50+. So most of the disagreement is ties among rare words.
- **The bands agree in 82.2% of words and are the same or neighbouring in 99.2%.**

**Next step:** use `targets.simple` (it counts entities, which is what "how common is this name" means); `names.txt` would give almost the same bands.

In [ ]:
# 3) largest disagreements among words that are common in at least one source
c = cmp[(cmp["n_entities"] >= 20) | (cmp["n_names_txt"] >= 20)].copy()
c["log_ratio"] = np.log10((c["n_names_txt"] + 1) / (c["n_entities"] + 1))
print("much more frequent in names.txt than in targets (spellings, positions, other schemas):")
display(c.sort_values("log_ratio", ascending=False).head(15)[["token", "n_entities", "n_names_txt"]])
print("much more frequent in targets than in names.txt:")
display(c.sort_values("log_ratio").head(15)[["token", "n_entities", "n_names_txt"]])

**Conclusion.**
- **Words much more frequent in `names.txt`** are position titles (councillor, deputy, mayor, secretary) and security names (obligatsii, serii): other schemas, not people's names.
- **Words more frequent in `targets.simple`** are alias-only words (siwy, gruby, polkovnik) and transliteration fragments.
- **Both confirm that `targets.simple` is the better source** for person and company names.

**Next step:** none; documented in `docs/decisions.md` (2026-11-13).

## 5 · Does it explain false alarms?

For each pair, two values:
- **rarest shared word:** frequency of the rarest word both sides share. A shared rare word ("torosov") is strong evidence; a shared common word ("ali") is weak.
- **rarest shared full name:** frequency of the rarest full name (order-insensitive key) both sides share; "none" if they share no full name.

⚠️ Runs on the **uncleaned** `pairs_scoped.parquet`: self-pairs inflate the positives. Re-run after the cleaning.
⚠️ `targets.simple` is the state *after* OpenSanctions merged duplicates: a name shared by a negative pair counts ≥ 2, by a positive pair ≥ 1. That is why 1 and 2 share one band below. The effect is strongest for full names.

In [ ]:
pairs = pd.read_parquet(C.PAIRS_SCOPED, columns=["pair_id", "judgement", "left_schema",
                                                 "left_name", "left_alias", "right_name", "right_alias"])

def side_tokens(names, aliases):
    return {t for n in list(names) + list(aliases) for t in tokens(n)}

def side_keys(names, aliases):
    return {k for k in (name_key(n) for n in list(names) + list(aliases)) if k}

t0 = time.time()
rows = []
for r in pairs.itertuples(index=False):
    group = "person" if r.left_schema == "Person" else "org"
    shared_words = side_tokens(r.left_name, r.left_alias) & side_tokens(r.right_name, r.right_alias)
    shared_names = side_keys(r.left_name, r.left_alias) & side_keys(r.right_name, r.right_alias)
    rarest_word = min((lookup[group].get(t, 0) for t in shared_words), default=np.nan)
    rarest_name = min((full_lookup[group].get(k, 0) for k in shared_names), default=np.nan)
    rows.append((r.pair_id, r.judgement, group == "person",
                 len(shared_words), rarest_word, len(shared_names), rarest_name))
res = pd.DataFrame(rows, columns=["pair_id", "judgement", "person",
                                  "n_shared_words", "rarest_word", "n_shared_names", "rarest_name"])
print(f"{len(res):,} pairs in {time.time() - t0:.0f}s")
print(f"no shared word: {res['rarest_word'].isna().mean():.1%}   no shared full name: {res['rarest_name'].isna().mean():.1%}")

In [ ]:
word_bands = [-1, 0, 2, 10, 100, 1000, 10**7]
word_labels = ["0", "1-2", "3-10", "11-100", "101-1000", ">1000"]
res["word_band"] = pd.cut(res["rarest_word"], word_bands, labels=word_labels)

full_bands = [-1, 0, 2, 5, 20, 10**7]
full_labels = ["0", "1-2", "3-5", "6-20", ">20"]
res["name_band"] = (pd.cut(res["rarest_name"], full_bands, labels=full_labels)
                      .cat.add_categories("none").fillna("none"))

def neg_share_table(s, col):
    tab = pd.crosstab(s[col], s["judgement"])
    tab["neg_share"] = (tab["negative"] / tab.sum(axis=1)).round(3)
    return tab

for person, title in [(True, "PERSON"), (False, "COMPANY / ORG")]:
    s = res[res["person"] == person]
    print(f"{title} · by rarest shared WORD"); display(neg_share_table(s, "word_band"))
    print(f"{title} · by rarest shared FULL NAME"); display(neg_share_table(s, "name_band"))

**Conclusion.**
- **The rarer the shared word, the more likely the pair is positive.** For persons, the negative share rises from 0.9% (word in 1–2 entities) to 80% (> 1,000 entities); companies show the same, from 4.8% to 96%.
- **The full-name bands are even sharper:** shared full name carried by 1–2 entities → 1.4% negative, 6–20 → 90%, > 20 → 97%.
- **No shared full name** (41% of pairs) → 87% negative.

**Next step:** keep both features and check whether the full name adds something beyond the word (next table).

### Does the full name add something beyond the rarest word?

Share of negative pairs for each combination of word band (rows) and full-name band (columns), persons only. If the share changes across a row, the full name explains false alarms that the word frequency alone does not.

In [ ]:
s = res[res["person"]]
counts = pd.crosstab(s["word_band"], s["name_band"])
neg = pd.crosstab(s["word_band"], s["name_band"], values=(s["judgement"] == "negative"), aggfunc="mean").round(3)
print("number of pairs"); display(counts)
print("share negative"); display(neg)

**Conclusion.**
- **Yes, the full name adds information.** Within the same word band, the full-name band changes the negative share strongly. Example: word band 11–100 has 2.7% negatives if the shared full name is rare (1–2), but 97.6% if it is carried by 6–20 entities.
- **So both features stay.** Band 0 (not in `targets.simple`) behaves like 1–2 (2–6% negative), which supports merging them.

**Next step:** use both as features in notebook 04.

## Open questions

- Stop list for company words (ltd, llc, ooo …) — needed, or does "rarest shared word" already ignore them?
- Features as `log1p` or as bands? Bands absorb the ±1 merge effect; for full names bands seem necessary.
- Keep both features (rarest word, rarest full name), or is one enough? See the combination table.
- Mean over shared words as an extra feature?
- Re-run section 5 after the cleaning.
- Log the collapse rule in `docs/decisions.md`: same name + same birth date = one person (about 34,000 person entities removed); the source-based rule was rejected.